# LiteOPM-Net

An explainable, lightweight multi-task CNN for joint **modulation format identification (MFI)** and
**OSNR estimation** from constellation images in coherent optical links.

This notebook contains the full pipeline:

1. a coherent link simulator (split-step Fourier fibre model, EDFA spans, ASE noise loading, receiver DSP) and its physics checks
2. generation of the constellation-image dataset (6 formats, OSNR 10–30 dB, 9 link conditions)
3. training of LiteOPM-Net and three baselines (AlexNet-style CNN, MobileNetV2, ResNet-18), 3 seeds each
4. Grad-CAM analysis with a deletion test and a region analysis
5. result tables and figures

**To run:** choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
On a free T4 GPU it takes about 45–75 minutes with the provided dataset, or about 1 hour more if you regenerate it.

## 1. Settings

In [ ]:
# True  = generate the dataset with the simulator (about 1 hour)
# False = upload the provided dataset (LiteOPM-Net_dataset.zip from the GitHub Releases page)
REGENERATE_DATA = False

# True = keep all work in Google Drive (MyDrive/LiteOPM-Net) so progress survives a disconnect
SAVE_TO_DRIVE = False

## 2. Setup

In [ ]:
import os, glob, json, time, subprocess, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else
      "none (training will be slow; use Runtime > Change runtime type > T4 GPU)")
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/LiteOPM-Net"
else:
    WORK = "/content/LiteOPM-Net"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print("Working folder:", WORK)

## 3. Source code
Each cell below writes one module to the working folder.

| File | Purpose |
|---|---|
| `fibre_sim.py` | coherent link simulator |
| `validate_sim.py` | physics checks of the simulator |
| `make_dataset.py`, `run_full_dataset.sh` | dataset generation |
| `data.py` | data loading and the fixed train/validation/test split |
| `models.py` | LiteOPM-Net and the baselines |
| `train.py` | training of one model |
| `xai.py` | Grad-CAM, deletion test and region analysis |
| `analyze.py`, `fig_schematics.py`, `fig_gradcam.py` | tables and figures |

In [ ]:
%%writefile fibre_sim.py
"""
fibre_sim.py - a small, self-contained coherent optical link simulator
(numpy only) for generating MFI / OSNR-monitoring datasets.

Signal chain
------------
Tx   : random symbols (6 formats) + periodic QPSK pilots
       -> root-raised-cosine (RRC) pulse shaping, 4 samples/symbol
       -> laser phase noise (Wiener process, combined Tx+LO linewidth)
Fibre: N spans of SSMF, each solved with the symmetric split-step Fourier
       method on the scalar nonlinear Schrodinger equation (loss,
       chromatic dispersion, Kerr nonlinearity), followed by an EDFA that
       restores the span loss
Rx   : ASE noise loading to set the OSNR exactly (0.1 nm reference)
       -> ideal chromatic-dispersion compensation (frequency domain)
       -> RRC matched filter, downsample to 1 sample/symbol
       -> pilot-aided carrier phase recovery (format-agnostic)
       -> AGC, drop pilots and edge symbols

Simplifications:
  - single polarisation (scalar NLSE), single channel (no WDM / XPM)
  - ASE added once at the receiver (noise loading) rather than per span
  - ideal timing recovery

Units: time in s, length in km, power in W.
"""

import numpy as np
from scipy.fft import next_fast_len

C =299792458.0             # speed of light (m/s)
LAMBDA = 1550e-9            # carrier wavelength (m)
B_REF = 12.5e9              # OSNR reference bandwidth, 0.1 nm (Hz)

FORMATS = ["QPSK", "8PSK", "8QAM", "16QAM", "32QAM", "64QAM"]


# ================================================================ formats
def _square_qam(m):
    k = int(np.sqrt(m))
    lv = np.arange(-(k - 1), k, 2)
    i, q = np.meshgrid(lv, lv)
    return (i + 1j * q).ravel()


def constellation(name):
    if name == "QPSK":
        c = np.exp(1j * (np.pi / 4 + np.pi / 2 * np.arange(4)))
    elif name == "8PSK":
        c = np.exp(1j * np.pi / 4 * np.arange(8))
    elif name == "8QAM":                        # star 8QAM, ring ratio 1+sqrt(3)
        c = np.concatenate([
            np.exp(1j * np.pi / 2 * np.arange(4)),
            (1 + np.sqrt(3)) * np.exp(1j * (np.pi / 4 + np.pi / 2 * np.arange(4))),
        ])
    elif name == "16QAM":
        c = _square_qam(16)
    elif name == "32QAM":                       # cross 32QAM
        lv = np.arange(-5, 6, 2)
        i, q = np.meshgrid(lv, lv)
        p = (i + 1j * q).ravel()
        c = p[~((np.abs(p.real) == 5) & (np.abs(p.imag) == 5))]
    elif name == "64QAM":
        c = _square_qam(64)
    else:
        raise ValueError(name)
    return c / np.sqrt(np.mean(np.abs(c) ** 2))


# ================================================================ filters
def rrc_freq(n, fs, baud, rolloff):
    """Frequency response of a root-raised-cosine filter (zero phase)."""
    f = np.abs(np.fft.fftfreq(n, 1 / fs))
    f1 = (1 - rolloff) * baud / 2
    f2 = (1 + rolloff) * baud / 2
    h = np.zeros(n)
    h[f <= f1] = 1.0
    mid = (f > f1) & (f <= f2)
    h[mid] = np.sqrt(0.5 * (1 + np.cos(np.pi / (rolloff * baud) * (f[mid] - f1))))
    return h


# ================================================================ link
class Link:
    def __init__(self, baud=32e9, sps=4, rolloff=0.1,
                 n_spans=1, span_km=80.0, alpha_db=0.2, d_ps_nm_km=17.0,
                 gamma=1.3, launch_dbm=2.0, step_km=1.0,
                 linewidth_hz=100e3, pilot_every=32, cpe_window=9):
        self.baud, self.sps, self.rolloff = baud, sps, rolloff
        self.fs = baud * sps
        self.n_spans, self.span_km = n_spans, span_km
        self.alpha = alpha_db / (10 * np.log10(np.e))           # 1/km (power)
        d = d_ps_nm_km * 1e-12 / (1e-9 * 1e3)                    # s/m^2
        self.beta2 = -d * LAMBDA ** 2 / (2 * np.pi * C) * 1e3   # s^2/km
        self.gamma = gamma                                       # 1/W/km
        self.p_launch = 1e-3 * 10 ** (launch_dbm / 10)          # W
        self.step_km = step_km
        self.linewidth = linewidth_hz
        self.pilot_every = pilot_every
        self.cpe_window = cpe_window

    @property
    def length_km(self):
        return self.n_spans * self.span_km

    # ---------------------------------------------------------- transmitter
    def transmit(self, fmt, n_data, rng):
        """Return the Tx field, the full symbol sequence and a data mask."""
        n_sym = n_data
        while n_sym - (-(-n_sym // self.pilot_every)) < n_data:
            n_sym += 1                                  # make room for pilots
        n_sym = next_fast_len(n_sym)                   # fast FFT length
        is_pilot = np.zeros(n_sym, bool)
        is_pilot[:: self.pilot_every] = True
        c_data = constellation(fmt)
        c_pil = constellation("QPSK")
        sym = np.empty(n_sym, complex)
        sym[is_pilot] = c_pil[rng.integers(0, 4, is_pilot.sum())]
        sym[~is_pilot] = c_data[rng.integers(0, len(c_data), (~is_pilot).sum())]

        up = np.zeros(n_sym * self.sps, complex)
        up[:: self.sps] = sym
        n = up.size
        x = np.fft.ifft(np.fft.fft(up) * rrc_freq(n, self.fs, self.baud, self.rolloff))
        x *= np.sqrt(self.p_launch / np.mean(np.abs(x) ** 2))

        if self.linewidth > 0:
            var = 2 * np.pi * self.linewidth / self.fs
            x *= np.exp(1j * np.cumsum(rng.normal(0, np.sqrt(var), n)))
        return x, sym, is_pilot

    # ---------------------------------------------------------- fibre
    def propagate(self, x):
        n = x.size
        w = 2 * np.pi * np.fft.fftfreq(n, 1 / self.fs)
        n_steps = int(round(self.span_km / self.step_km))
        dz = self.span_km / n_steps
        half_lin = np.exp((-self.alpha / 2 + 0.5j * self.beta2 * w ** 2) * dz / 2)
        for _ in range(self.n_spans):
            for _ in range(n_steps):                    # symmetric split-step
                x = np.fft.ifft(np.fft.fft(x) * half_lin)
                x = x * np.exp(1j * self.gamma * np.abs(x) ** 2 * dz)
                x = np.fft.ifft(np.fft.fft(x) * half_lin)
            x = x * np.exp(self.alpha * self.span_km / 2)  # EDFA: restore loss
        return x

    # ---------------------------------------------------------- receiver
    def add_ase(self, x, osnr_db, rng):
        """Noise loading. Scalar field = one polarisation of a DP signal,
        so OSNR = P_pol / (N0 * B_ref) with N0 the ASE PSD per polarisation."""
        p = np.mean(np.abs(x) ** 2)
        n0 = p / (10 ** (osnr_db / 10) * B_REF)
        sigma = np.sqrt(n0 * self.fs / 2)
        return x + sigma * (rng.normal(size=x.size) + 1j * rng.normal(size=x.size))

    def receive(self, y, sym, is_pilot):
        n = y.size
        w = 2 * np.pi * np.fft.fftfreq(n, 1 / self.fs)
        cd_comp = np.exp(-0.5j * self.beta2 * w ** 2 * self.length_km)
        y = np.fft.ifft(np.fft.fft(y) * cd_comp
                        * rrc_freq(n, self.fs, self.baud, self.rolloff))
        r = y[:: self.sps]
        r = r / np.sqrt(np.mean(np.abs(r) ** 2))

        # pilot-aided carrier phase recovery (does not need the data format)
        idx = np.flatnonzero(is_pilot)
        z = r[idx] * np.conj(sym[idx])
        k = np.ones(self.cpe_window) / self.cpe_window
        z_pad = np.concatenate([z[-self.cpe_window:], z, z[: self.cpe_window]])
        z_s = np.convolve(z_pad, k, mode="same")[self.cpe_window:-self.cpe_window]
        ph = np.unwrap(np.angle(z_s))
        phase = np.interp(np.arange(r.size), idx, ph)
        r = r * np.exp(-1j * phase)

        d = r[~is_pilot]
        return d / np.sqrt(np.mean(np.abs(d) ** 2)), sym[~is_pilot]

    # ---------------------------------------------------------- full chain
    def run(self, fmt, osnr_db, n_data, rng, edge=64):
        """Returns recovered data symbols, transmitted data symbols and
        the effective SNR (dB) measured from EVM."""
        x, sym, is_pilot = self.transmit(fmt, n_data + 2 * edge, rng)
        y = self.add_ase(self.propagate(x), osnr_db, rng)
        r, s = self.receive(y, sym, is_pilot)
        r, s = r[edge:edge + n_data], s[edge:edge + n_data]
        # best complex scaling, then EVM -> effective SNR
        g = np.vdot(s, r) / np.vdot(s, s)
        err = r / g - s
        snr_eff = 10 * np.log10(np.mean(np.abs(s) ** 2) / np.mean(np.abs(err) ** 2))
        return r, s, snr_eff


def osnr_to_snr_db(osnr_db, baud=32e9):
    """Theoretical SNR per symbol after matched filtering (ASE only)."""
    return osnr_db + 10 * np.log10(B_REF / baud)

In [ ]:
%%writefile validate_sim.py
"""
Physics checks for fibre_sim.py, saved to validation.json. Each test compares
the simulator with a result known from theory.

  1. Back-to-back: measured SNR must match SNR = OSNR + 10log10(Bref/Rs)
  2. Linear fibre (gamma = 0): after CD compensation the signal must be
     recovered to the noise-only limit, for 80 km and 800 km
  3. Nonlinear fibre, no ASE: nonlinear SNR must fall ~2 dB per 1 dB
     increase in launch power (GN-model behaviour)
  4. SSFM step size: result must not depend on step (0.5, 1, 2 km)
"""
import json
import numpy as np
from fibre_sim import Link, osnr_to_snr_db

rng = np.random.default_rng(1)
N = 8192
out = {}

print("1) Back-to-back noise loading (16QAM, no fibre, no phase noise)")
b2b = []
for o in [10, 15, 20, 25, 30]:
    _, _, snr = Link(n_spans=0, linewidth_hz=0).run("16QAM", o, N, rng)
    b2b.append(dict(osnr=o, theory=osnr_to_snr_db(o), measured=snr))
    print(f"   OSNR {o:2d} dB -> theory {osnr_to_snr_db(o):5.2f}, measured {snr:5.2f} dB")
out["b2b"] = b2b
out["b2b_max_err"] = max(abs(r["theory"] - r["measured"]) for r in b2b)

print("2) Linear fibre, CD compensation (64QAM, gamma=0, OSNR 60 dB)")
lin = []
for spans in [1, 10]:
    _, _, snr = Link(n_spans=spans, gamma=0.0, linewidth_hz=0).run("64QAM", 60, N, rng)
    lin.append(dict(km=spans * 80, snr=snr, limit=osnr_to_snr_db(60)))
    print(f"   {spans*80:4d} km -> SNR {snr:5.1f} dB (limit {osnr_to_snr_db(60):.1f})")
out["linear"] = lin

print("3) Nonlinear penalty vs launch power (16QAM, 800 km, OSNR 60 dB)")
nl = []
for p in [0, 2, 4, 6, 8]:
    _, _, snr = Link(n_spans=10, launch_dbm=p, linewidth_hz=0).run("16QAM", 60, N, rng)
    nl.append(dict(launch_dbm=p, snr=snr))
    print(f"   launch {p:+d} dBm -> SNR {snr:5.1f} dB")
out["nonlinear"] = nl
slopes = [(nl[i + 1]["snr"] - nl[i]["snr"]) / 2 for i in range(len(nl) - 1)]
out["nl_slope_db_per_db"] = [min(slopes), max(slopes)]

print("4) SSFM step size (16QAM, 800 km, 4 dBm)")
st = []
for step in [0.5, 1.0, 2.0]:
    r = np.random.default_rng(5)
    _, _, snr = Link(n_spans=10, launch_dbm=4, step_km=step, linewidth_hz=0).run("16QAM", 60, N, r)
    st.append(dict(step_km=step, snr=snr))
    print(f"   step {step} km -> SNR {snr:.3f} dB")
out["step"] = st
out["step_spread_db"] = max(s["snr"] for s in st) - min(s["snr"] for s in st)

json.dump(out, open("validation.json", "w"), indent=1)
print(json.dumps({k: out[k] for k in ("b2b_max_err", "nl_slope_db_per_db", "step_spread_db")}))

In [ ]:
%%writefile make_dataset.py
"""
Generate an MFI / OSNR dataset of constellation images with fibre_sim.py.

Pilot:   python make_dataset.py --spans 1 --per_class 30 --out pilot_ssfm.npz
Full set (later): loop over --spans 1 2 5 10 and --launch_dbm -2 0 2 4
                  and concatenate, keeping some settings unseen for testing.
"""
import argparse
import json
import os
from multiprocessing import Pool

import numpy as np
from fibre_sim import FORMATS, Link

N_SYM = 4096            # data symbols per image
IMG = 64
AXIS = 1.9              # plot range (+/-) on unit-power scale
OSNR_DB = np.arange(10, 31, 1)


def to_image(y):
    h, _, _ = np.histogram2d(y.real, y.imag, bins=IMG,
                             range=[[-AXIS, AXIS], [-AXIS, AXIS]])
    h = np.log1p(h)
    h = h / h.max() if h.max() > 0 else h
    return (h.T[::-1] * 255).astype(np.uint8)


def job(args):
    fi, osnr, seed, link_kw = args
    rng = np.random.default_rng(seed)
    r, _, snr = Link(**link_kw).run(FORMATS[fi], osnr, N_SYM, rng)
    return to_image(r), fi, osnr, snr


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--spans", type=int, default=1, help="80 km spans")
    ap.add_argument("--launch_dbm", type=float, default=2.0)
    ap.add_argument("--linewidth_khz", type=float, default=100.0)
    ap.add_argument("--per_class", type=int, default=30)
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--workers", type=int, default=os.cpu_count())
    ap.add_argument("--out", default="pilot_ssfm.npz")
    a = ap.parse_args()

    link_kw = dict(n_spans=a.spans, launch_dbm=a.launch_dbm,
                   linewidth_hz=a.linewidth_khz * 1e3)
    jobs, s = [], a.seed
    for fi in range(len(FORMATS)):
        for o in OSNR_DB:
            for _ in range(a.per_class):
                jobs.append((fi, float(o), s, link_kw)); s += 1

    with Pool(a.workers) as p:
        res = p.map(job, jobs, chunksize=8)

    img, fmt, osnr, snr = zip(*res)
    lk = Link(**link_kw)
    meta = dict(formats=FORMATS, osnr_db=OSNR_DB.tolist(),
                fibre_km=lk.length_km, launch_dbm=a.launch_dbm,
                linewidth_hz=lk.linewidth, baud=lk.baud,
                pilot_every=lk.pilot_every, cpe_window=lk.cpe_window,
                n_sym_per_image=N_SYM, image_size=IMG, axis_range=AXIS,
                per_class=a.per_class, seed=a.seed,
                model="scalar NLSE split-step, EDFA spans, ASE noise loading, "
                      "ideal CD comp, RRC matched filter, pilot-aided CPE")
    np.savez_compressed(a.out, images=np.stack(img),
                        fmt=np.array(fmt, np.int8),
                        osnr=np.array(osnr, np.float32),
                        snr_eff=np.array(snr, np.float32),
                        meta=json.dumps(meta))
    print(f"Saved {len(img)} images ({lk.length_km:.0f} km, "
          f"{a.launch_dbm} dBm) to {a.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile run_full_dataset.sh
#!/bin/bash
# Full dataset for the LiteOPM-Net experiments.
# Seen conditions (train / validation / test split later):
#   80, 240, 400 km  x  launch 0 and 2 dBm, 100 kHz linewidth
# Unseen conditions (robustness test only, never used in training):
#   U1: 160 km, 1 dBm            (interpolation in length and power)
#   U2: 560 km, 3 dBm            (extrapolation in length and power)
#   U3:  80 km, 2 dBm, 1 MHz     (unseen, cheaper lasers)
set -e
cd "$(dirname "$0")"
mkdir -p data
PC=10   # images per (format, OSNR) per condition -> 1260 per condition

run () {
  if [ -f data/$1.npz ]; then echo "skip $1 (done)"; return; fi
  echo "$(date +%T) start $1"
  python3 make_dataset.py "${@:2}" --per_class $PC --out data/$1.tmp.npz && mv data/$1.tmp.npz data/$1.npz
}

run seen_80km_0dBm   --spans 1 --launch_dbm 0 --seed 1000
run seen_80km_2dBm   --spans 1 --launch_dbm 2 --seed 2000
run seen_240km_0dBm  --spans 3 --launch_dbm 0 --seed 3000
run seen_240km_2dBm  --spans 3 --launch_dbm 2 --seed 4000
run seen_400km_0dBm  --spans 5 --launch_dbm 0 --seed 5000
run seen_400km_2dBm  --spans 5 --launch_dbm 2 --seed 6000
run unseen_160km_1dBm --spans 2 --launch_dbm 1 --seed 7000
run unseen_560km_3dBm --spans 7 --launch_dbm 3 --seed 8000
run unseen_80km_2dBm_1MHz --spans 1 --launch_dbm 2 --linewidth_khz 1000 --seed 9000
echo "$(date +%T) ALL DONE"

In [ ]:
%%writefile data.py
"""Dataset loading and splitting for the LiteOPM-Net experiments."""
import glob
import json
import os

import numpy as np

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "data")
OSNR_MEAN, OSNR_SCALE = 20.0, 10.0          # target normalisation for regression


def load(pattern):
    """Load and concatenate every npz in data/ matching the glob pattern."""
    files = sorted(f for f in glob.glob(os.path.join(DATA_DIR, pattern))
                   if not f.endswith(".tmp.npz"))
    if not files:
        raise FileNotFoundError(pattern)
    X, F, O, S, C = [], [], [], [], []
    for ci, f in enumerate(files):
        d = np.load(f)
        X.append(d["images"]); F.append(d["fmt"]); O.append(d["osnr"])
        S.append(d["snr_eff"]); C.append(np.full(len(d["fmt"]), ci))
        meta = json.loads(str(d["meta"]))
    names = [os.path.basename(f)[:-4] for f in files]
    return dict(x=np.concatenate(X), fmt=np.concatenate(F).astype(np.int64),
                osnr=np.concatenate(O).astype(np.float32),
                snr=np.concatenate(S).astype(np.float32),
                cond=np.concatenate(C), cond_names=names,
                formats=meta["formats"])


def split(d, seed=0, frac=(0.70, 0.10, 0.20)):
    """Stratified split by (condition, format, OSNR)."""
    rng = np.random.default_rng(seed)
    key = d["cond"] * 10000 + d["fmt"] * 100 + d["osnr"].astype(int)
    tr, va, te = [], [], []
    for k in np.unique(key):
        idx = rng.permutation(np.flatnonzero(key == k))
        n_tr = int(round(frac[0] * len(idx)))
        n_va = int(round(frac[1] * len(idx)))
        tr += list(idx[:n_tr]); va += list(idx[n_tr:n_tr + n_va])
        te += list(idx[n_tr + n_va:])
    return np.array(tr), np.array(va), np.array(te)

In [ ]:
%%writefile models.py
"""
Models for joint modulation format identification (MFI) and OSNR estimation.

Every model returns (logits[B, 6], osnr_norm[B]). Single-task variants
return the unused output as zeros so one training loop serves all.

  LiteOPMNet   proposed: lightweight depthwise-separable CNN, shared
               backbone with two heads (task = "both" | "mfi" | "osnr")
  AlexStyleCNN classic 5-conv CNN in the style of early CNN-based optical
               performance monitoring work
  ResNet18MT   ResNet-18 (torchvision) with 1-channel input and two heads
  MobileNetV2MT MobileNetV2 (torchvision) with 1-channel input and two heads
"""
import torch
import torch.nn as nn


class DSBlock(nn.Module):
    """Depthwise 3x3 -> BN -> ReLU -> pointwise 1x1 -> BN -> ReLU."""

    def __init__(self, c_in, c_out, stride):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_in, 3, stride, 1, groups=c_in, bias=False),
            nn.BatchNorm2d(c_in), nn.ReLU(inplace=True),
            nn.Conv2d(c_in, c_out, 1, bias=False),
            nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.net(x)


class TwoHeads(nn.Module):
    def __init__(self, c, n_cls, task):
        super().__init__()
        self.task = task
        self.mfi = nn.Linear(c, n_cls) if task in ("both", "mfi") else None
        self.osnr = (nn.Sequential(nn.Linear(c, 64), nn.ReLU(inplace=True),
                                   nn.Linear(64, 1))
                     if task in ("both", "osnr") else None)
        self.n_cls = n_cls

    def forward(self, h):
        b = h.shape[0]
        logits = self.mfi(h) if self.mfi is not None else h.new_zeros(b, self.n_cls)
        osnr = self.osnr(h).squeeze(1) if self.osnr is not None else h.new_zeros(b)
        return logits, osnr


class LiteOPMNet(nn.Module):
    def __init__(self, n_cls=6, task="both", width=(16, 32, 64, 96, 128)):
        super().__init__()
        w = width
        self.stem = nn.Sequential(nn.Conv2d(1, w[0], 3, 1, 1, bias=False),
                                  nn.BatchNorm2d(w[0]), nn.ReLU(inplace=True))
        self.features = nn.Sequential(
            DSBlock(w[0], w[1], 2),     # 64 -> 32
            DSBlock(w[1], w[2], 2),     # 32 -> 16
            DSBlock(w[2], w[3], 1),     # 16
            DSBlock(w[3], w[4], 1))     # 16  (Grad-CAM layer, 16x16 maps)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.heads = TwoHeads(w[4], n_cls, task)

    @property
    def cam_layer(self):
        return self.features[-1]

    def forward(self, x):
        h = self.pool(self.features(self.stem(x))).flatten(1)
        return self.heads(h)


class AlexStyleCNN(nn.Module):
    def __init__(self, n_cls=6, task="both"):
        super().__init__()
        def blk(i, o, pool):
            layers = [nn.Conv2d(i, o, 3, 1, 1), nn.BatchNorm2d(o), nn.ReLU(inplace=True)]
            if pool:
                layers.append(nn.MaxPool2d(2))
            return layers
        self.features = nn.Sequential(*blk(1, 32, True), *blk(32, 64, True),
                                      *blk(64, 128, False), *blk(128, 128, False),
                                      *blk(128, 128, True))          # 8x8
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(128 * 8 * 8, 256),
                                nn.ReLU(inplace=True), nn.Dropout(0.3))
        self.heads = TwoHeads(256, n_cls, task)

    @property
    def cam_layer(self):
        return self.features[-2]

    def forward(self, x):
        return self.heads(self.fc(self.features(x)))


class ResNet18MT(nn.Module):
    def __init__(self, n_cls=6, task="both"):
        super().__init__()
        from torchvision.models import resnet18
        m = resnet18(weights=None)
        m.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        m.fc = nn.Identity()
        self.body = m
        self.heads = TwoHeads(512, n_cls, task)

    @property
    def cam_layer(self):
        return self.body.layer4

    def forward(self, x):
        return self.heads(self.body(x))


class MobileNetV2MT(nn.Module):
    def __init__(self, n_cls=6, task="both"):
        super().__init__()
        from torchvision.models import mobilenet_v2
        m = mobilenet_v2(weights=None)
        m.features[0][0] = nn.Conv2d(1, 32, 3, 2, 1, bias=False)
        m.classifier = nn.Identity()
        self.body = m
        self.heads = TwoHeads(1280, n_cls, task)

    @property
    def cam_layer(self):
        return self.body.features[-1]

    def forward(self, x):
        return self.heads(self.body(x))


MODELS = {"liteopm": LiteOPMNet, "alexstyle": AlexStyleCNN,
          "resnet18": ResNet18MT, "mobilenetv2": MobileNetV2MT}


def build(name, task="both"):
    return MODELS[name](task=task)


def count_params(m):
    return sum(p.numel() for p in m.parameters())


def count_macs(m, shape=(1, 1, 64, 64)):
    """Multiply-accumulates of conv and linear layers for one input."""
    macs = []

    def conv_hook(mod, inp, out):
        k = mod.kernel_size[0] * mod.kernel_size[1] * mod.in_channels // mod.groups
        macs.append(out.numel() * k)

    def lin_hook(mod, inp, out):
        macs.append(mod.in_features * mod.out_features)

    hs = []
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            hs.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.Linear):
            hs.append(mod.register_forward_hook(lin_hook))
    m.eval()
    with torch.no_grad():
        m(torch.zeros(shape))
    for h in hs:
        h.remove()
    return int(sum(macs))

In [ ]:
%%writefile train.py
"""
Train one model and save predictions on the seen test split and on every
unseen condition.

    python train.py --model liteopm --task both --seed 0
    python train.py --model resnet18 --task both --seed 0

Output: runs/<model>_<task>_s<seed>/{best.pt, history.json, preds.npz, info.json}
"""
import argparse
import json
import os
import time

import numpy as np
import torch
import torch.nn.functional as F

from data import OSNR_MEAN, OSNR_SCALE, load, split
from models import build, count_macs, count_params

HERE = os.path.dirname(os.path.abspath(__file__))


def tensors(d, idx):
    x = torch.from_numpy(d["x"][idx].astype(np.float32) / 255.0).unsqueeze(1)
    f = torch.from_numpy(d["fmt"][idx])
    o = torch.from_numpy((d["osnr"][idx] - OSNR_MEAN) / OSNR_SCALE)
    return x, f, o


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


@torch.no_grad()
def predict(model, x, bs=256):
    model.eval()
    L, O = [], []
    for i in range(0, len(x), bs):
        l, o = model(x[i:i + bs].to(DEVICE))
        L.append(l.cpu()); O.append(o.cpu())
    logits = torch.cat(L)
    return logits.argmax(1).numpy(), (torch.cat(O) * OSNR_SCALE + OSNR_MEAN).numpy(), \
        torch.softmax(logits, 1).numpy()


def score(task, acc, mae):
    """Lower is better; used for picking the best epoch on validation."""
    if task == "mfi":
        return 1 - acc
    if task == "osnr":
        return mae
    return (1 - acc) + mae / 5.0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="liteopm")
    ap.add_argument("--task", default="both", choices=["both", "mfi", "osnr"])
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--bs", type=int, default=64)
    ap.add_argument("--lr", type=float, default=2e-3)
    ap.add_argument("--lam", type=float, default=1.0, help="OSNR loss weight")
    ap.add_argument("--threads", type=int, default=2)
    a = ap.parse_args()

    torch.manual_seed(a.seed); np.random.seed(a.seed)
    torch.set_num_threads(a.threads)
    out = os.path.join(HERE, "runs", f"{a.model}_{a.task}_s{a.seed}")
    os.makedirs(out, exist_ok=True)

    d = load("seen_*.npz")
    tr, va, te = split(d, seed=0)               # same split for every model
    xtr, ftr, otr = [t.to(DEVICE) for t in tensors(d, tr)]
    print(f"device: {DEVICE}", flush=True)
    xva, fva, ova = tensors(d, va)
    xte, fte, ote = tensors(d, te)

    model = build(a.model, a.task).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-4)
    steps = a.epochs * int(np.ceil(len(tr) / a.bs))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=steps,
                                                pct_start=0.15)
    use_cls = a.task in ("both", "mfi")
    use_reg = a.task in ("both", "osnr")

    hist, best, t0 = [], np.inf, time.time()
    for ep in range(a.epochs):
        model.train()
        perm = torch.randperm(len(tr))
        tot = 0.0
        for i in range(0, len(tr), a.bs):
            b = perm[i:i + a.bs].to(DEVICE)
            logits, o = model(xtr[b])
            loss = 0.0
            if use_cls:
                loss = loss + F.cross_entropy(logits, ftr[b])
            if use_reg:
                loss = loss + a.lam * F.l1_loss(o, otr[b])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item() * len(b)
        pf, po, _ = predict(model, xva)
        acc = float((pf == fva.numpy()).mean())
        mae = float(np.abs(po - d["osnr"][va]).mean())
        s = score(a.task, acc, mae)
        hist.append(dict(epoch=ep + 1, loss=tot / len(tr), val_acc=acc, val_mae=mae,
                         time=time.time() - t0))
        print(f"ep {ep+1:2d} loss {tot/len(tr):.4f} val_acc {acc*100:.2f}% "
              f"val_mae {mae:.3f} dB  [{time.time()-t0:.0f}s]", flush=True)
        if s < best:
            best = s
            torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()},
                       os.path.join(out, "best.pt"))

    model.load_state_dict(torch.load(os.path.join(out, "best.pt"), map_location=DEVICE))
    preds = {}
    pf, po, pp = predict(model, xte)
    preds.update(seen_fmt_pred=pf, seen_osnr_pred=po, seen_prob=pp,
                 seen_fmt=d["fmt"][te], seen_osnr=d["osnr"][te],
                 seen_cond=d["cond"][te], seen_idx=te)
    un = load("unseen_*.npz")
    xu = torch.from_numpy(un["x"].astype(np.float32) / 255.0).unsqueeze(1)
    pf, po, pp = predict(model, xu)
    preds.update(unseen_fmt_pred=pf, unseen_osnr_pred=po, unseen_prob=pp,
                 unseen_fmt=un["fmt"], unseen_osnr=un["osnr"], unseen_cond=un["cond"])
    np.savez_compressed(os.path.join(out, "preds.npz"), **preds)

    train_time = time.time() - t0
    model = model.cpu()                          # MAC counting runs on CPU
    info = dict(model=a.model, task=a.task, seed=a.seed, epochs=a.epochs, lr=a.lr,
                lam=a.lam, bs=a.bs, params=count_params(model),
                macs=count_macs(model), train_time_s=train_time, device=str(DEVICE),
                seen_conditions=d["cond_names"], unseen_conditions=un["cond_names"],
                n_train=len(tr), n_val=len(va), n_test=len(te))
    json.dump(hist, open(os.path.join(out, "history.json"), "w"), indent=1)
    json.dump(info, open(os.path.join(out, "info.json"), "w"), indent=1)
    print(json.dumps(info))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile xai.py
"""
Grad-CAM analysis of the trained multi-task LiteOPM-Net.

1. Grad-CAM maps for the MFI head (predicted-class logit) and the OSNR head
   (evidence for higher OSNR = +output, evidence for lower OSNR = -output)
2. Physics check: share of each map's mass on
     core  - within 1 sigma of an ideal constellation point
     halo  - between 1 and 2.5 sigma (the noise spread)
     gaps  - everywhere else
   sigma is the per-dimension ASE noise std from the measured SNR.
3. Faithfulness (deletion test): remove the top-k% pixels ranked by
   Grad-CAM vs. at random and track accuracy / OSNR error.

    python xai.py --run runs/liteopm_both_s0
"""
import argparse
import json
import os

import numpy as np
import torch
import torch.nn.functional as F

from data import OSNR_MEAN, OSNR_SCALE, load
from fibre_sim import constellation
from models import build

AXIS, IMG = 1.9, 64


def gradcam(model, x, target):
    """target(logits, osnr) -> per-sample scalar. Returns CAMs [B,64,64] in [0,1]."""
    acts, grads = {}, {}
    layer = model.cam_layer
    h1 = layer.register_forward_hook(lambda m, i, o: acts.__setitem__("a", o))
    h2 = layer.register_full_backward_hook(lambda m, gi, go: grads.__setitem__("g", go[0]))
    model.eval()
    x = x.clone().requires_grad_(True)
    logits, osnr = model(x)
    model.zero_grad()
    target(logits, osnr).sum().backward()
    h1.remove(); h2.remove()
    a, g = acts["a"].detach(), grads["g"].detach()
    w = g.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(IMG, IMG), mode="bilinear", align_corners=False)[:, 0]
    mx = cam.flatten(1).max(1).values.clamp_min(1e-12)
    return (cam / mx[:, None, None]).numpy()


def region_masks(fmt_name, snr_db):
    """Boolean masks (core, halo, gaps) on the 64x64 image grid."""
    snr = 10 ** (snr_db / 10)
    ps = snr / (1 + snr)                       # signal share after receiver AGC
    sigma = np.sqrt((1 - ps) / 2)
    pts = constellation(fmt_name) * np.sqrt(ps)
    centres = -AXIS + (np.arange(IMG) + 0.5) * (2 * AXIS / IMG)
    re = centres[None, :].repeat(IMG, 0)
    im = centres[::-1][:, None].repeat(IMG, 1)  # row 0 = top = +imag
    z = re + 1j * im
    dist = np.min(np.abs(z[..., None] - pts[None, None, :]), axis=-1)
    core = dist <= sigma
    halo = (dist > sigma) & (dist <= 2.5 * sigma)
    return core, halo, ~(core | halo)


def mass_shares(cam, masks):
    tot = cam.sum() + 1e-12
    return [float(cam[m].sum() / tot) for m in masks]


@torch.no_grad()
def eval_batch(model, x, fmt, osnr):
    model.eval()
    logits, o = model(x)
    acc = float((logits.argmax(1).numpy() == fmt).mean())
    mae = float(np.abs(o.numpy() * OSNR_SCALE + OSNR_MEAN - osnr).mean())
    return acc, mae


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run", default="runs/liteopm_both_s0")
    ap.add_argument("--n", type=int, default=600, help="test images used")
    a = ap.parse_args()
    torch.set_num_threads(1)

    info = json.load(open(os.path.join(a.run, "info.json")))
    model = build(info["model"], info["task"])
    model.load_state_dict(torch.load(os.path.join(a.run, "best.pt")))
    p = np.load(os.path.join(a.run, "preds.npz"))
    d = load("seen_*.npz")
    rng = np.random.default_rng(0)
    idx = np.sort(rng.choice(p["seen_idx"], size=min(a.n, len(p["seen_idx"])), replace=False))
    x = torch.from_numpy(d["x"][idx].astype(np.float32) / 255.0).unsqueeze(1)
    fmt, osnr, snr = d["fmt"][idx], d["osnr"][idx], d["snr"][idx]
    names = d["formats"]

    with torch.no_grad():
        pred = model(x)[0].argmax(1)
    cam_mfi = gradcam(model, x, lambda l, o: l.gather(1, pred[:, None])[:, 0])
    cam_hi = gradcam(model, x, lambda l, o: o)
    cam_lo = gradcam(model, x, lambda l, o: -o)

    # ---- physics check: where does each map put its mass?
    rows = []
    for i in range(len(idx)):
        m = region_masks(names[fmt[i]], float(snr[i]))
        area = [float(mm.mean()) for mm in m]
        rows.append(dict(fmt=int(fmt[i]), osnr=float(osnr[i]), area=area,
                         mfi=mass_shares(cam_mfi[i], m), hi=mass_shares(cam_hi[i], m),
                         lo=mass_shares(cam_lo[i], m)))

    def summarise(sel):
        r = [rows[i] for i in sel]
        return {k: np.mean([q[k] for q in r], axis=0).round(3).tolist()
                for k in ("area", "mfi", "hi", "lo")}

    phys = {"all": summarise(range(len(rows))),
            "osnr_ge_18": summarise([i for i, q in enumerate(rows) if q["osnr"] >= 18]),
            "osnr_le_14": summarise([i for i, q in enumerate(rows) if q["osnr"] <= 14])}

    # ---- deletion test
    fracs = np.arange(0, 0.55, 0.05)
    flat = x.view(len(x), -1)
    del_res = {}
    for name, cam in [("mfi", cam_mfi), ("osnr", np.maximum(cam_hi, cam_lo))]:
        order = np.argsort(-cam.reshape(len(x), -1), axis=1)
        rnd = np.argsort(rng.random(order.shape), axis=1)
        curves = {"gradcam": [], "random": []}
        for f in fracs:
            k = int(round(f * IMG * IMG))
            for tag, ordr in (("gradcam", order), ("random", rnd)):
                xx = flat.clone()
                if k:
                    rows_i = np.repeat(np.arange(len(x)), k)
                    xx[rows_i, ordr[:, :k].ravel()] = 0.0
                curves[tag].append(eval_batch(model, xx.view_as(x), fmt, osnr))
        del_res[name] = {t: np.array(v).tolist() for t, v in curves.items()}

    out = dict(physics=phys, deletion_fracs=fracs.round(2).tolist(), deletion=del_res,
               n_images=len(idx))
    json.dump(out, open(os.path.join(a.run, "xai.json"), "w"), indent=1)
    np.savez_compressed(os.path.join(a.run, "cams.npz"), idx=idx, cam_mfi=cam_mfi,
                        cam_hi=cam_hi, cam_lo=cam_lo, fmt=fmt, osnr=osnr, snr=snr)
    print(json.dumps(phys, indent=1))
    for name in del_res:
        print(name, "gradcam:", np.round(del_res[name]["gradcam"], 3)[:, 0 if name == "mfi" else 1])
        print(name, "random: ", np.round(del_res[name]["random"], 3)[:, 0 if name == "mfi" else 1])


if __name__ == "__main__":
    main()

In [ ]:
%%writefile analyze.py
"""
Aggregate all training runs into summary tables (results.json) and
figures (figs/). Run after train.py runs and xai.py have finished.
"""
import glob
import json
import os
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch

from data import load
from models import build

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
                     "font.size": 8, "axes.linewidth": 0.6, "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False})
HERE = os.path.dirname(os.path.abspath(__file__))
FIG = os.path.join(HERE, "figs")
os.makedirs(FIG, exist_ok=True)
NAMES = {"liteopm": "LiteOPM-Net (proposed)", "alexstyle": "AlexNet-style CNN",
         "resnet18": "ResNet-18", "mobilenetv2": "MobileNetV2"}
COLORS = {"liteopm": "#c0392b", "alexstyle": "#7f8c8d", "resnet18": "#2e6da4",
          "mobilenetv2": "#27865f"}
ORDER = ["liteopm", "alexstyle", "mobilenetv2", "resnet18"]
UNSEEN_LABEL = {"unseen_160km_1dBm": "160 km, 1 dBm",
                "unseen_560km_3dBm": "560 km, 3 dBm",
                "unseen_80km_2dBm_1MHz": "80 km, 1 MHz laser"}


def metrics(fp, op, f, o):
    err = op - o
    return dict(acc=float((fp == f).mean()), mae=float(np.abs(err).mean()),
                rmse=float(np.sqrt((err ** 2).mean())))


def latency_ms(name, task="both", n=300):
    torch.set_num_threads(1)
    m = build(name, task).eval()
    x = torch.rand(1, 1, 64, 64)
    with torch.no_grad():
        for _ in range(30):
            m(x)
        t = time.perf_counter()
        for _ in range(n):
            m(x)
    return (time.perf_counter() - t) / n * 1e3


def mean_std(vals):
    v = np.array(vals, float)
    return [float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else 0.0]


def main():
    runs = {}
    for p in sorted(glob.glob(os.path.join(HERE, "runs", "*", "info.json"))):
        info = json.load(open(p))
        pr = dict(np.load(os.path.join(os.path.dirname(p), "preds.npz")))
        runs.setdefault((info["model"], info["task"]), []).append((info, pr))

    res = {"models": {}, "ablation": {}, "per_osnr": {}, "unseen": {}}
    osnr_grid = np.arange(10, 31)
    for (m, task), lst in runs.items():
        seen = [metrics(pr["seen_fmt_pred"], pr["seen_osnr_pred"], pr["seen_fmt"],
                        pr["seen_osnr"]) for _, pr in lst]
        low = [metrics(pr["seen_fmt_pred"][pr["seen_osnr"] <= 14],
                       pr["seen_osnr_pred"][pr["seen_osnr"] <= 14],
                       pr["seen_fmt"][pr["seen_osnr"] <= 14],
                       pr["seen_osnr"][pr["seen_osnr"] <= 14]) for _, pr in lst]
        entry = dict(n_seeds=len(lst), params=lst[0][0]["params"], macs=lst[0][0]["macs"],
                     train_time_s=mean_std([i["train_time_s"] for i, _ in lst]),
                     acc=mean_std([s["acc"] for s in seen]),
                     acc_low=mean_std([s["acc"] for s in low]),
                     mae=mean_std([s["mae"] for s in seen]),
                     rmse=mean_std([s["rmse"] for s in seen]))
        if task == "both":
            entry["latency_ms"] = latency_ms(m)
            res["models"][m] = entry
            # accuracy and MAE vs OSNR (pooled over seeds)
            fp = np.concatenate([pr["seen_fmt_pred"] for _, pr in lst])
            f = np.concatenate([pr["seen_fmt"] for _, pr in lst])
            op = np.concatenate([pr["seen_osnr_pred"] for _, pr in lst])
            o = np.concatenate([pr["seen_osnr"] for _, pr in lst])
            res["per_osnr"][m] = dict(
                acc=[float((fp[o == k] == f[o == k]).mean()) for k in osnr_grid],
                mae=[float(np.abs(op[o == k] - k).mean()) for k in osnr_grid])
            # unseen conditions
            cond_names = lst[0][0]["unseen_conditions"]
            un = {}
            for ci, cn in enumerate(cond_names):
                ms = [metrics(pr["unseen_fmt_pred"][pr["unseen_cond"] == ci],
                              pr["unseen_osnr_pred"][pr["unseen_cond"] == ci],
                              pr["unseen_fmt"][pr["unseen_cond"] == ci],
                              pr["unseen_osnr"][pr["unseen_cond"] == ci]) for _, pr in lst]
                un[cn] = dict(acc=mean_std([q["acc"] for q in ms]),
                              mae=mean_std([q["mae"] for q in ms]))
            res["unseen"][m] = un
        if m == "liteopm":
            res["ablation"][task] = entry
    # OSNR bias (pred - true) vs OSNR on unseen links, proposed model, pooled seeds
    lst = runs[("liteopm", "both")]
    res["unseen_bias"] = {}
    for ci, cn in enumerate(lst[0][0]["unseen_conditions"]):
        o = np.concatenate([pr["unseen_osnr"][pr["unseen_cond"] == ci] for _, pr in lst])
        op = np.concatenate([pr["unseen_osnr_pred"][pr["unseen_cond"] == ci] for _, pr in lst])
        res["unseen_bias"][cn] = dict(
            bias=[float((op[o == k] - k).mean()) for k in osnr_grid],
            mae=[float(np.abs(op[o == k] - k).mean()) for k in osnr_grid],
            mae_le20=float(np.abs(op[o <= 20] - o[o <= 20]).mean()),
            mae_gt20=float(np.abs(op[o > 20] - o[o > 20]).mean()))
    # SNR saturation per unseen condition (from data) for the discussion
    un = load("unseen_*.npz")
    res["snr_at_30dB"] = {cn: float(un["snr"][(un["cond"] == ci) & (un["osnr"] == 30)].mean())
                          for ci, cn in enumerate(un["cond_names"])}
    sd = load("seen_*.npz")
    res["snr_at_30dB"].update({cn: float(sd["snr"][(sd["cond"] == ci) & (sd["osnr"] == 30)].mean())
                               for ci, cn in enumerate(sd["cond_names"])})
    # per-format accuracy and MAE, proposed model pooled over seeds
    fp = np.concatenate([pr["seen_fmt_pred"] for _, pr in lst])
    f = np.concatenate([pr["seen_fmt"] for _, pr in lst])
    op = np.concatenate([pr["seen_osnr_pred"] for _, pr in lst])
    o = np.concatenate([pr["seen_osnr"] for _, pr in lst])
    res["per_format"] = {n: dict(acc=float((fp[f == k] == k).mean()),
                                 mae=float(np.abs(op[f == k] - o[f == k]).mean()))
                         for k, n in enumerate(sd["formats"])}
    res["first_osnr_100"] = {m: next((int(k) for k, a in zip(osnr_grid, v["acc"])
                                      if all(x >= 0.999 for x in v["acc"][int(k) - 10:])), None)
                             for m, v in res["per_osnr"].items()}
    xai_p = os.path.join(HERE, "runs", "liteopm_both_s0", "xai.json")
    if os.path.exists(xai_p):
        xr = json.load(open(xai_p))
        fr = np.array(xr["deletion_fracs"])
        auc = lambda y: float(np.trapezoid(y, fr) / (fr[-1] - fr[0]))
        res["deletion_auc"] = {
            "mfi_acc": {t: auc(np.array(xr["deletion"]["mfi"][t])[:, 0]) for t in ("gradcam", "random")},
            "osnr_mae": {t: auc(np.array(xr["deletion"]["osnr"][t])[:, 1]) for t in ("gradcam", "random")}}
    json.dump(res, open(os.path.join(HERE, "results.json"), "w"), indent=1)

    # ---------------------------------------------------------- Fig. 4
    d = load("seen_*.npz")
    names = d["formats"]
    _, pr = runs[("liteopm", "both")][0]
    fig, ax = plt.subplots(1, 3, figsize=(7.0, 2.25),
                           gridspec_kw=dict(width_ratios=[1.15, 1.0, 1.0]))
    for m in ORDER:
        if m in res["per_osnr"]:
            ax[0].plot(osnr_grid, np.array(res["per_osnr"][m]["acc"]) * 100, "-o", ms=2.2,
                       lw=1.1, color=COLORS[m], label=NAMES[m].replace(" (proposed)", "").replace(" CNN", ""))
    ax[0].set_xlabel("OSNR (dB)"); ax[0].set_ylabel("MFI accuracy (%)")
    ax[0].set_ylim(top=100.5); ax[0].legend(fontsize=6.0, loc="lower right", handlelength=1.4, borderaxespad=0.2)
    ax[0].set_title("(a) Format accuracy vs OSNR", fontsize=8)
    lowm = pr["seen_osnr"] <= 14
    cm = np.zeros((6, 6))
    for t, p in zip(pr["seen_fmt"][lowm], pr["seen_fmt_pred"][lowm]):
        cm[t, p] += 1
    cmn = cm / cm.sum(1, keepdims=True) * 100
    ax[1].imshow(cmn, cmap="Blues", vmin=0, vmax=100)
    for i in range(6):
        for j in range(6):
            if cm[i, j]:
                ax[1].text(j, i, f"{cmn[i, j]:.0f}", ha="center", va="center", fontsize=6,
                           color="white" if cmn[i, j] > 60 else "#1f2328")
    short = [n.replace("QAM", "Q").replace("PSK", "P") for n in names]
    ax[1].set_xticks(range(6)); ax[1].set_xticklabels(short, fontsize=6.3)
    ax[1].set_yticks(range(6)); ax[1].set_yticklabels(short, fontsize=6.3)
    ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("True")
    ax[1].set_title("(b) Confusion, OSNR 10-14 dB (%)", fontsize=8)
    for s in ("top", "right"):
        ax[1].spines[s].set_visible(True)
    jit = np.random.default_rng(0).uniform(-0.25, 0.25, len(pr["seen_osnr"]))
    ax[2].scatter(pr["seen_osnr"] + jit, pr["seen_osnr_pred"], s=1.2, alpha=0.35,
                  color=COLORS["liteopm"], lw=0)
    ax[2].plot([9, 31], [9, 31], color="#1f2328", lw=0.7, ls="--")
    ax[2].set_xlim(9, 31); ax[2].set_ylim(9, 31)
    ax[2].set_xlabel("True OSNR (dB)"); ax[2].set_ylabel("Estimated OSNR (dB)")
    ax[2].set_title("(c) OSNR estimation", fontsize=8)
    fig.tight_layout(pad=0.3, w_pad=1.2)
    fig.savefig(os.path.join(FIG, "fig4_results.png"), dpi=300)

    # ---------------------------------------------------------- Fig. 6
    xai_p = os.path.join(HERE, "runs", "liteopm_both_s0", "xai.json")
    if os.path.exists(xai_p):
        xr = json.load(open(xai_p))
        fr = np.array(xr["deletion_fracs"]) * 100
        fig, ax = plt.subplots(1, 3, figsize=(7.0, 2.2))
        for tag, ls in (("gradcam", "-"), ("random", "--")):
            ax[0].plot(fr, np.array(xr["deletion"]["mfi"][tag])[:, 0] * 100, ls, marker="o",
                       ms=2.2, lw=1.1, color=COLORS["liteopm"] if tag == "gradcam" else "#7f8c8d",
                       label="Grad-CAM order" if tag == "gradcam" else "Random order")
            ax[1].plot(fr, np.array(xr["deletion"]["osnr"][tag])[:, 1], ls, marker="o",
                       ms=2.2, lw=1.1, color=COLORS["liteopm"] if tag == "gradcam" else "#7f8c8d")
        ax[0].set_xlabel("Pixels removed (%)"); ax[0].set_ylabel("MFI accuracy (%)")
        ax[0].legend(fontsize=6.5); ax[0].set_title("(a) Deletion test, MFI", fontsize=8)
        ax[1].set_xlabel("Pixels removed (%)"); ax[1].set_ylabel("OSNR MAE (dB)")
        ax[1].set_title("(b) Deletion test, OSNR", fontsize=8)
        conds = list(UNSEEN_LABEL)
        w = 0.2
        for k, m in enumerate(ORDER):
            if m not in res["unseen"]:
                continue
            ys = [res["unseen"][m][c]["mae"][0] for c in conds]
            es = [res["unseen"][m][c]["mae"][1] for c in conds]
            ax[2].bar(np.arange(3) + (k - 1.5) * w, ys, w, yerr=es, color=COLORS[m],
                      label=NAMES[m].replace(" (proposed)", "").replace(" CNN", ""), error_kw=dict(lw=0.6, capsize=1.5))
        ax[2].set_xticks(range(3))
        ax[2].set_xticklabels([UNSEEN_LABEL[c].replace(", ", ",\n") for c in conds], fontsize=6.3)
        ax[2].set_ylabel("OSNR MAE (dB)"); ax[2].set_ylim(0, 4.2)
        ax[2].legend(fontsize=5.8, loc="upper left", ncol=2, handlelength=1.0, columnspacing=0.8)
        ax[2].set_title("(c) Unseen link conditions", fontsize=8)
        fig.tight_layout(pad=0.3, w_pad=1.0)
        fig.savefig(os.path.join(FIG, "fig6_faithful_robust.png"), dpi=300)
    print(json.dumps(res["models"], indent=1))
    print(json.dumps(res["ablation"], indent=1))
    print(json.dumps(res["unseen"], indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile fig_schematics.py
"""Schematic figures: simulated link (Fig. 1) and LiteOPM-Net (Fig. 2)."""
import os
import matplotlib
matplotlib.use("Agg")
os.makedirs("figs", exist_ok=True)
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
                     "font.size": 8})
INK, EDGE = "#1f2328", "#57606a"
TX, FIB, RX, NET, HEAD = "#dbe9f6", "#e3f1df", "#fbe7d3", "#ece4f6", "#f6e1e1"


def box(ax, x, y, w, h, text, fc, fs=7.2):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.06",
                                fc=fc, ec=EDGE, lw=0.8))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=fs, color=INK)


def arrow(ax, x0, y0, x1, y1):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", color=EDGE, lw=0.8, mutation_scale=8))


# ------------------------------------------------------------ Fig. 1 link
fig, ax = plt.subplots(figsize=(7.0, 1.9))
ax.set_xlim(0, 14); ax.set_ylim(0, 3.6); ax.axis("off")
w, h = 1.55, 1.0
xs = [0.1, 1.95, 3.8, 5.65]
labels = ["Symbols +\nQPSK pilots\n(1 in 32)", "RRC shaping\n(roll-off 0.1,\n4 Sa/sym)",
          "Laser phase\nnoise\n(Wiener)", "SSMF span\n(SSFM: loss,\nCD, Kerr)"]
fcs = [TX, TX, TX, FIB]
for x, t, c in zip(xs, labels, fcs):
    box(ax, x, 2.2, w, h, t, c)
for a, b in zip(xs[:-1], xs[1:]):
    arrow(ax, a + w, 2.7, b, 2.7)
box(ax, 7.5, 2.2, 1.2, h, "EDFA", FIB)
arrow(ax, 5.65 + w, 2.7, 7.5, 2.7)
ax.annotate("", xy=(5.65, 3.35), xytext=(8.7, 3.35),
            arrowprops=dict(arrowstyle="-|>", color=EDGE, lw=0.8, ls="--",
                            connectionstyle="arc3,rad=0", mutation_scale=8))
ax.text(7.2, 3.47, "x N spans (80 km each)", ha="center", fontsize=7, color=INK)
box(ax, 9.15, 2.2, 1.55, h, "ASE noise\nloading\n(sets OSNR)", RX)
arrow(ax, 8.7, 2.7, 9.15, 2.7)
# receiver row
rx = [(9.15, "CD\ncompensation"), (7.3, "Matched\nRRC filter,\n1 Sa/sym"),
      (5.45, "Pilot-aided\nphase\nrecovery"), (3.6, "64x64\nconstellation\nimage"),
      (1.75, "LiteOPM-Net")]
for i, (x, t) in enumerate(rx):
    box(ax, x, 0.25, 1.55, h, t, NET if i == 4 else RX)
arrow(ax, 9.9, 2.2, 9.9, 1.25)
for (a, _), (b, _) in zip(rx[:-1], rx[1:]):
    arrow(ax, a, 0.75, b + 1.55, 0.75)
ax.text(0.95, 1.0, "Format\n(6 classes)", ha="center", fontsize=7, color=INK)
ax.text(0.95, 0.25, "OSNR\n(dB)", ha="center", fontsize=7, color=INK)
arrow(ax, 1.75, 0.95, 1.45, 1.15); arrow(ax, 1.75, 0.55, 1.45, 0.4)
ax.text(11.0, 2.7, "Transmitter and fibre link", fontsize=7.5, style="italic", va="center")
ax.text(11.0, 0.75, "Coherent receiver DSP\nand monitor", fontsize=7.5, style="italic",
        va="center")
fig.tight_layout(pad=0.1)
fig.savefig("figs/fig1_link.png", dpi=300)

# ------------------------------------------------------------ Fig. 2 model
fig, ax = plt.subplots(figsize=(7.0, 1.75))
ax.set_xlim(0, 14); ax.set_ylim(0, 3.2); ax.axis("off")
blocks = [("Input\n1x64x64", TX), ("Conv 3x3\n16 ch", NET),
          ("DS block\n32 ch, s2\n32x32", NET), ("DS block\n64 ch, s2\n16x16", NET),
          ("DS block\n96 ch\n16x16", NET), ("DS block\n128 ch\n16x16", NET),
          ("Global\navg pool\n128", NET)]
x, bw, step = 0.05, 1.32, 1.6
for i, (t, c) in enumerate(blocks):
    box(ax, x, 1.15, bw, 1.25, t, c, fs=7)
    if i:
        arrow(ax, x - (step - bw), 1.77, x, 1.77)
    x += step
cam_x = 0.05 + 5 * step + bw / 2
ax.text(cam_x, 2.72, "Grad-CAM layer", ha="center", fontsize=6.8, color="#9a3b3b")
ax.annotate("", xy=(cam_x, 2.44), xytext=(cam_x, 2.68),
            arrowprops=dict(arrowstyle="-|>", color="#9a3b3b", lw=0.7, mutation_scale=6))
pool_r = 0.05 + 6 * step + bw
box(ax, 11.9, 2.0, 2.0, 0.9, "MFI head\nFC-6, softmax", HEAD, fs=7)
box(ax, 11.9, 0.65, 2.0, 0.9, "OSNR head\nFC-64, FC-1", HEAD, fs=7)
arrow(ax, pool_r, 1.9, 11.9, 2.45); arrow(ax, pool_r, 1.65, 11.9, 1.1)
ax.text(0.05, 0.3, "DS block = depthwise 3x3 conv, BN, ReLU, pointwise 1x1 conv, BN, ReLU."
        "   Loss = cross-entropy + L1(OSNR).", ha="left", fontsize=6.8, color=INK)
fig.tight_layout(pad=0.1)
fig.savefig("figs/fig2_model.png", dpi=300)
print("ok")

In [ ]:
%%writefile fig_gradcam.py
"""Fig. 5: Grad-CAM examples for both heads + Fig. 3: example constellations."""
import os
import matplotlib
matplotlib.use("Agg")
os.makedirs("figs", exist_ok=True)
import matplotlib.pyplot as plt
import numpy as np

from data import load

plt.rcParams.update({"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
                     "font.size": 8})
d = load("seen_*.npz")
names = d["formats"]

# ------------------------------------------------------------ Fig. 3
c = d["cond_names"].index("seen_400km_2dBm")
osnrs = [12, 20, 30]
fig, ax = plt.subplots(len(osnrs), len(names), figsize=(7.0, 3.65))
for r, o in enumerate(osnrs):
    for k, n in enumerate(names):
        i = np.flatnonzero((d["fmt"] == k) & (d["osnr"] == o) & (d["cond"] == c))[0]
        a = ax[r, k]
        a.imshow(d["x"][i], cmap="magma"); a.set_xticks([]); a.set_yticks([])
        if r == 0:
            a.set_title(n, fontsize=7.5)
        if k == 0:
            a.set_ylabel(f"OSNR {o} dB", fontsize=7.5)
fig.tight_layout(pad=0.2, h_pad=0.25, w_pad=0.15)
fig.savefig("figs/fig3_constellations.png", dpi=300)

# ------------------------------------------------------------ Fig. 5
c = np.load("runs/liteopm_both_s0/cams.npz")
pick = []
for fname, o_lo, o_hi in [("8QAM", 16, 22), ("16QAM", 18, 24), ("64QAM", 22, 28)]:
    fi = names.index(fname)
    cand = np.flatnonzero((c["fmt"] == fi) & (c["osnr"] >= o_lo) & (c["osnr"] <= o_hi))
    pick.append(cand[len(cand) // 2])
cols = ["Input", "MFI head", "OSNR head:\nraises estimate", "OSNR head:\nlowers estimate"]
fig, ax = plt.subplots(3, 4, figsize=(3.45, 2.75))
for r, j in enumerate(pick):
    img = d["x"][c["idx"][j]] / 255.0
    maps = [None, c["cam_mfi"][j], c["cam_hi"][j], c["cam_lo"][j]]
    for k in range(4):
        a = ax[r, k]
        a.imshow(img, cmap="gray_r" if k else "magma", alpha=1.0 if k == 0 else 0.55)
        if k:
            a.imshow(maps[k], cmap="jet", alpha=0.5, vmin=0, vmax=1)
        a.set_xticks([]); a.set_yticks([])
        if r == 0:
            a.set_title(cols[k], fontsize=6.3)
        if k == 0:
            a.set_ylabel(f"{names[c['fmt'][j]]}\n{c['osnr'][j]:.0f} dB", fontsize=6.5)
fig.tight_layout(pad=0.15, h_pad=0.2, w_pad=0.15)
fig.savefig("figs/fig5_gradcam.png", dpi=300)
print("ok", [int(c["osnr"][j]) for j in pick])

## 4. Check the simulator (about 3 minutes)
Compares the simulator with known theory: SNR versus OSNR without fibre, dispersion compensation,
the nonlinear penalty versus launch power, and step-size convergence.

In [ ]:
!python validate_sim.py

## 5. Dataset

In [ ]:
os.makedirs("data", exist_ok=True)
if REGENERATE_DATA:
    !bash run_full_dataset.sh
else:
    if not glob.glob("data/*.npz"):
        if not os.path.exists("LiteOPM-Net_dataset.zip"):
            from google.colab import files
            print("Please upload LiteOPM-Net_dataset.zip")
            for name in files.upload():
                if name != "LiteOPM-Net_dataset.zip" and name.startswith("LiteOPM-Net_dataset"):
                    os.replace(name, "LiteOPM-Net_dataset.zip")
        !unzip -oq LiteOPM-Net_dataset.zip
print("Dataset files:", sorted(os.path.basename(f) for f in glob.glob("data/*.npz")))

## 6. Train all models (about 15–60 minutes on a T4 GPU)
18 runs: LiteOPM-Net (joint, MFI-only and OSNR-only) and the three baselines, each with 3 seeds.
Finished runs are skipped, so this cell can be re-run after a disconnect.

In [ ]:
jobs = [("liteopm", t, s, "2e-3") for s in (0, 1, 2) for t in ("both", "mfi", "osnr")]
jobs += [(m, "both", s, "1e-3") for s in (0, 1, 2) for m in ("mobilenetv2", "alexstyle", "resnet18")]
t_all = time.time()
for k, (m, task, s, lr) in enumerate(jobs, 1):
    run = f"runs/{m}_{task}_s{s}"
    if os.path.exists(f"{run}/info.json"):
        print(f"[{k:2d}/{len(jobs)}] skip {run} (done)")
        continue
    os.makedirs(run, exist_ok=True)
    t = time.time()
    with open(f"{run}/train.log", "w") as log:
        r = subprocess.run(["python", "train.py", "--model", m, "--task", task, "--seed", str(s),
                            "--lr", lr, "--threads", "2"], stdout=log, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        print(open(f"{run}/train.log").read()[-2000:])
        raise RuntimeError(f"{run} failed")
    last = [l for l in open(f"{run}/train.log") if l.startswith("ep 30")]
    print(f"[{k:2d}/{len(jobs)}] done {run} in {time.time()-t:.0f}s  {last[0].strip() if last else ''}")
print(f"All training finished in {(time.time()-t_all)/60:.0f} min")

## 7. Grad-CAM analysis, tables and figures (about 3 minutes)

In [ ]:
!python xai.py --run runs/liteopm_both_s0 --n 600 > xai_log.txt && echo "Grad-CAM done"
!python analyze.py > analyze_log.txt && echo "Tables done"
!python fig_schematics.py > /dev/null && python fig_gradcam.py > /dev/null && echo "Figures done"

## 8. Results

In [ ]:
res = json.load(open("results.json"))
NAMES = {"liteopm": "LiteOPM-Net", "alexstyle": "AlexNet-style CNN", "mobilenetv2": "MobileNetV2",
         "resnet18": "ResNet-18"}
pm = lambda v, s=1, d=2: f"{v[0]*s:.{d}f} ± {v[1]*s:.{d}f}"
print("Seen test set (mean ± std over 3 seeds)")
print(f"{'Model':20s}{'Params':>10s}{'MFI acc (%)':>16s}{'acc 10-14 dB (%)':>19s}{'OSNR MAE (dB)':>16s}{'CPU ms':>8s}")
for m, n in NAMES.items():
    e = res["models"][m]
    print(f"{n:20s}{e['params']:>10,d}{pm(e['acc'],100,1):>16s}{pm(e['acc_low'],100,1):>19s}"
          f"{pm(e['mae'],1,2):>16s}{e['latency_ms']:>8.2f}")
print("\nJoint vs single-task LiteOPM-Net")
a = res["ablation"]
print(f"  joint: acc {pm(a['both']['acc'],100,1)}%, MAE {pm(a['both']['mae'])} dB")
print(f"  MFI-only acc {pm(a['mfi']['acc'],100,1)}%,  OSNR-only MAE {pm(a['osnr']['mae'])} dB")
print("\nUnseen link conditions (OSNR MAE, dB)")
for m, n in NAMES.items():
    print(f"  {n:20s}" + "  ".join(f"{c[7:]}: {pm(v['mae'])}" for c, v in res["unseen"][m].items()))

## 9. Figures

In [ ]:
from IPython.display import Image, display
for f in ["fig3_constellations.png", "fig4_results.png", "fig5_gradcam.png", "fig6_faithful_robust.png"]:
    print(f); display(Image(f"figs/{f}", width=900))

## 10. Download results

In [ ]:
from google.colab import files
!zip -qr LiteOPM-Net_results.zip results.json validation.json figs runs/*/info.json runs/*/train.log runs/*/history.json xai_log.txt analyze_log.txt
files.download("LiteOPM-Net_results.zip")